# Training of Capacitive-Memristor Network

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import itertools
import random
import scipy
from joblib import Parallel, delayed

from functions import *

## Simulated Dynamics

The following code is used to initialize the ring graph that we will be using. There will be 4 modules, each containing about 100 nanowires, with an average degree of 10 (maximum of 20), and there will be at most 2 wires that connect the modules in a ring. 

In [ ]:
num_cluster = 4
cluster_size = 100
avg_degree = 10
max_degree = 20
max_connecting_nodes = 2

clusteredG = random_clustered_ring_graph(num_cluster,cluster_size,avg_degree,max_degree,max_connecting_nodes)

Now we can begin the Forward Euler integration scheme to simulate the network dynamics. There can be modifications made to decrease the amount of data needed (which will be shown in a different file), however this demonstration is meant to keep track of the wire (node) voltages, memristive (edge) currents, and memristive states. 

The following code is an example of adding a specific number of source (input) and drain nodes. Each input will have a time delay. 

In [ ]:
nx.set_edge_attributes(clusteredG,5e-2,name='x')

T = 20
dt = 1e-4

nt = int(T/dt)

t = np.linspace(0,T,nt)

voltage = 10*(np.sin(2*np.pi*t*1) + 0)

const_state = [1e-14,1e-8,10]
r = [100,16000]      # Note that the scaling is not the same

node_list = list(clusteredG.nodes())
edge_list = list(clusteredG.edges())

y = np.array([clusteredG.edges[n1,n2].get('x') for n1,n2 in edge_list])
y = np.expand_dims(y,axis=0)

x = np.zeros((len(edge_list), nt))
node_v = np.zeros((len(node_list), nt))

const_cap = [2e-12,95]
#const_cap = [0,0]

num_sources = 4
ran_sources = np.random.choice(node_list, num_sources, replace=False)
#source = 2
not_source_nodes = [node for node in node_list if node not in ran_sources]
print("Source Nodes:",ran_sources)
#print(not_source_nodes)

num_drain = 2
ran_drains = np.random.choice(not_source_nodes,num_drain,replace=False)
print("Drain Nodes:",ran_drains)
#drain = 77
'''
for node,attr in clusteredG.nodes(data=True):
    if node == source:        
        clusteredG.nodes[node]["Type"] = "Source"
        
    if node == drain:
        clusteredG.nodes[node]["Type"] = "Drain"
'''        
for node in ran_sources:
    clusteredG.nodes[node]["Type"] = "Source"
for node in ran_drains:
    clusteredG.nodes[node]["Type"] = "Drain"


v_shifted, shifts = shift(t,voltage,1,num_sources)
v_same = np.full((num_sources,nt),voltage)
print(v_shifted.shape,v_same.shape)

N = len(edge_list)
pert_step = 5e-2
max_pert = 5
idx_array = [i for i in range(len(const_state))]

const_state_array = param_network(N, const_state, pert_step, max_pert, idx_array)
r_array = param_network(N,r,pert_step, max_pert, [0,1])
#const_cap_array = param_network(N, const_cap, pert_step, max_pert, [0,1])
const_cap_array = param_network(N, const_cap, 0.0, max_pert, [0,1])
    
current = np.zeros_like(x)
mem_voltages =  np.zeros_like(x)

info = graph_info(clusteredG, const_state_array, r_array, const_cap_array)
q_prev = np.zeros((1,len(edge_list)))

# Initialize all possible arrays and values
# q_prev is a function of 
print(info["Source Nodes"])

for i in range(nt):
    v_input = v_shifted[:,i]
    #print(v_input)
    #print(v_input.shape)
    
    dy, curr, mem_voltage, q_new = evolution(t,dt,y,v_input,const_state_array,clusteredG,"HP Decay",r_array,node_list, q_prev, info)

    q_prev = q_new
    
    #y = y + dy*dt
    y = np.clip(y + dy*dt,0.0,1.0)
    x[:,i] = y.flatten()

    current[:,i] = curr.flatten()
    node_v[:,i] = np.array([clusteredG.nodes[node].get("Voltage", 0.0) for node in node_list])
    mem_voltages[:,i] = mem_voltage

The following are visualizations of the wire voltages, and to ensure that voltages of the drains and sources are behaving accordingly.

In [ ]:
# Network wire voltages
plt.figure()
plt.plot(voltage,node_v.T)
plt.title('Node Voltage Responses vs Input Voltage')
plt.ylabel('Node Voltage (V)')
plt.xlabel('Input Voltage (V)')
plt.show()

# Drain voltages
for idx,drain in enumerate(info["Drain Nodes"]):
    
    plt.figure()
    plt.plot(voltage,node_v[info["Node Indices"][drain],:])
    plt.title(f'Drain {drain} Voltage Response vs Input Voltage')
    plt.ylabel('Voltage (V)')
    plt.xlabel('Input Voltage (V)')
    plt.show()

# Source voltages
for idx,source in enumerate(info["Source Nodes"]):
    
    plt.figure()
    plt.plot(v_shifted[idx,:],node_v[info["Node Indices"][source],:])
    plt.title(f'Input Node {source} Voltage Response vs Input Voltage ')
    plt.ylabel('Voltage (V)')
    plt.xlabel('Input Voltage (V)')
    plt.show()

The following are I-V (current-voltage) plots of randomly picked memristive units.

In [ ]:
# Pick random edges and plot current vs voltage difference
ran_edges_idx = np.random.randint(0,len(info['Edge List']),4)

ran_edges = [info['Edge List'][i] for i in ran_edges_idx]
ran_curr = current[ran_edges_idx, :]

ran_v_diff = mem_voltages[ran_edges_idx, :] 

fig, axes = plt.subplots(2, 2, figsize=(10, 8), layout='constrained')
fig.suptitle('I-V of Memristor Edge', fontsize=16)
for idx, ax in enumerate(axes.flat):
    ax.plot(ran_v_diff[idx, :], ran_curr[idx])
    ax.set_title(f'{ran_edges[idx]}')
    ax.set_xlabel('Voltage (V)')
    ax.set_ylabel('Current (A)')

plt.show()

## Training of the Network

For the training, I treat the network as a Restricted Echo State Network, and use linear ridge regression to train. The goal is to perform a non-linear wave transformation, or obtaining some non-linear function of the original input signal.

In [ ]:
#sig = np.tanh(voltage)
sig = np.sign(voltage)
#sig = 10*(np.sign(np.sin(2*np.pi*t*5)) + 1)
#sig = np.arcsin(voltage/10)
#sig = np.cos(2*np.pi*t*2)
sig = np.expand_dims(sig,axis=1)

k = sig.shape[1]

sig = (sig - np.mean(sig))/np.std(sig)

#for i in range(k):
    #sig[:,i]=(sig[:,i]-np.mean(sig[:,i]))/np.std(sig[:,i]) 

print(node_v.shape)
print(sig.shape)

λ = 1e-5

basis_node = []
cluster_basis = []
cluster_nodes = []
for node,v in zip(node_list,node_v):
    #print(clusteredG.nodes[node].get("Cluster"))
    if node in ran_sources:
        continue
    elif node in ran_drains:
        continue
    else:
        if clusteredG.nodes[node]["Type"] == "Cluster" and clusteredG.nodes[node]["Cluster"] == 4:
            #print('added')
            cluster_basis.append(v)
            cluster_nodes.append(node)
        basis_node.append(v)

basis_node = np.array(basis_node)
cluster_basis = np.array(cluster_basis)
print(basis_node.shape)
# Current from cluster 4
clus_4_curr = []
for idx, edge in enumerate(edge_list):
    if edge[0] and edge[1] in cluster_nodes:
        #print(edge)
        clus_4_curr.append(current[idx])
clus_4_curr = np.array(clus_4_curr)
print("Cluster 4 Currents:",clus_4_curr.shape)
print(cluster_basis.shape)
print(current.shape)
#basis = basis_node[:,int(5/dt):int(10/dt)].T
basis = current[:,int(5/dt):int(10/dt)].T
#basis = clus_4_curr[:,int(5/dt):int(10/dt)].T

sup = sig[int(5/dt):int(10/dt),:]

phi = np.linalg.pinv(basis.T@basis+λ*np.eye(len(current)))@(basis.T@sup)


trained = phi.T@current

plt.figure(figsize=(10, 4))
plt.plot(t, sig, label='Supervisor', color='blue')
plt.plot(t, trained.T, label='Prediction', color='red', linestyle='--')
plt.xlabel("Time(s)")
plt.ylabel("Signal")
plt.title("Memristor Network Output (State-Dependent Current)")
plt.tight_layout()
plt.xlim(10,15)
plt.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)
plt.grid(True, linestyle="--", alpha=0.4)
plt.ylim(-1.5*np.max(sig),1.5*np.max(sig))
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(t, voltage, label='Supervisor', color='blue')
plt.xlabel("Time (s)")
plt.ylabel("Voltage (V)")
plt.title("Memristor Network Input")
plt.tight_layout()
plt.xlim(0,2)
plt.grid(True, linestyle="--", alpha=0.4)
#plt.ylim(-0.1*np.max(voltage(t)),1.1*np.max(voltage(t)))
plt.show()